# 12 · Small data files for the dashboard (Spark + Sedona)

**Run this in Google Colab after notebook 11.** It takes about 10–15 minutes.

The dashboard runs on your laptop, which cannot open 13 million roofs. This notebook summarises them with Spark
into small files (a few MB in total):
- **hexagons** (H3 resolution 7, about 5 km² each) with roofs, suitable share and solar potential,
- **districts** with totals for the technical maximum and the realistic case (at most 3 kW per good roof),
- a **random sample of 40,000 roofs** for the zoomed-in map,
- **district borders** (simplified) and the CSV results of notebooks 02–11.

Everything is zipped into `MyDrive/rooftop_solar/results/dashboard_data.zip` → download it to your laptop.

**Spark UI screenshots for the report:** click the link under Step 3b while Step 4 runs, and screenshot the
**Jobs**, **Stages** and **SQL / DataFrame** tabs.

**How to run:** Runtime → Run all. Allow Google Drive access.

In [ ]:
# Step 1: connect Google Drive, set folders, copy the predictions, free areas and district borders to the fast Colab disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
FEATURES = f'{PROJECT}/data/features'
RESULTS = f'{PROJECT}/results'
LOGS = f'{PROJECT}/logs'
LOCAL = '/content/work'      # fast Colab disk, deleted when the session ends
for folder in (RESULTS, LOGS, LOCAL):
    os.makedirs(folder, exist_ok=True)

LOG_FILE = f'{LOGS}/12_log.txt'
open(LOG_FILE, 'w').close()


def log(*parts):
    """Print, and also save to Google Drive so results survive if the tab closes."""
    text = ' '.join(str(p) for p in parts)
    print(text)
    with open(LOG_FILE, 'a') as f:
        f.write(text + '\n')


def log_table(pdf, csv_name=None):
    """Log a small pandas table; optionally save it as a CSV in results/."""
    log(pdf.to_string(index=False))
    if csv_name:
        pdf.to_csv(f'{RESULTS}/{csv_name}', index=False)


t = time.time()
for name in ('roof_predictions_gold', 'usable_area'):
    shutil.copytree(f'{FEATURES}/{name}', f'{LOCAL}/{name}', dirs_exist_ok=True)
shutil.copy(f'{PROJECT}/data/raw/kerala_boundaries.parquet', f'{LOCAL}/kerala_boundaries.parquet')
log(f'Inputs copied to {LOCAL} in {time.time() - t:.0f} s')

In [ ]:
# Step 2: install Java 17, PySpark 4.1, Apache Sedona 1.9.1 (about 2-5 minutes)
import subprocess


def java_version():
    java_bin = os.path.join(os.environ['JAVA_HOME'], 'bin', 'java') if 'JAVA_HOME' in os.environ else 'java'
    out = subprocess.run(f'{java_bin} -version 2>&1', shell=True, capture_output=True, text=True).stdout
    return out.splitlines()[0] if out else 'no java'


if not any(f'version "{v}' in java_version() for v in ('17', '21')):
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip install -q pyspark==4.1.3 apache-sedona==1.9.1
log('Java:', java_version())

# Sedona's Java files, downloaded directly. Spark 4.1 ships its own copy of the JTS geometry library,
# so Sedona must sit on the same class path as Spark (see Step 3) instead of using spark.jars.packages.
import urllib.request

JARS = '/content/jars'
MAVEN = 'https://repo1.maven.org/maven2'
os.makedirs(JARS, exist_ok=True)
JAR_FILES = []
for url in (f'{MAVEN}/org/apache/sedona/sedona-spark-shaded-4.1_2.13/1.9.1/sedona-spark-shaded-4.1_2.13-1.9.1.jar',
            f'{MAVEN}/org/datasyslab/geotools-wrapper/1.9.1-33.5/geotools-wrapper-1.9.1-33.5.jar'):
    path = f'{JARS}/{url.rsplit("/", 1)[1]}'
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    JAR_FILES.append(path)
log('Sedona JARs:', [os.path.basename(p) for p in JAR_FILES])

In [ ]:
# Step 3: start Spark with Sedona, using all CPU cores and most of the RAM
import psutil
from sedona.spark import SedonaContext

CORES = os.cpu_count()
MEM_GB = max(2, int(psutil.virtual_memory().total / 1e9 * 0.6))   # leave room for Python and the OS

config = (
    SedonaContext.builder()
    .master('local[*]')
    .appName('rooftop_dashboard')
    .config('spark.driver.memory', f'{MEM_GB}g')
    .config('spark.sql.shuffle.partitions', str(CORES * 4))
    .config('spark.driver.extraClassPath', ':'.join(JAR_FILES))
    .config('spark.sql.maxConcurrentOutputFileWriters', '20')   # write districts without sorting first
    .getOrCreate()
)
sedona = SedonaContext.create(config)
log(f'Spark {sedona.version} | CPU cores: {CORES} | Spark memory: {MEM_GB} GB')

In [ ]:
# Step 3b: open the Spark UI (click the link below while Step 4 runs, then screenshot "Jobs", "Stages" and "SQL")
from google.colab import output
output.serve_kernel_port_as_window(4040, path='/jobs/')

In [ ]:
# Step 4: one row per roof with its maximum and realistic solar, and its H3 hexagon (resolution 7)
import json
import pandas as pd

HOME_KW = 3
sedona.read.parquet(f'{LOCAL}/roof_predictions_gold').createOrReplaceTempView('pred')
sedona.read.parquet(f'{LOCAL}/usable_area').select('id', 'kwp_usable', 'kwh_year_usable').createOrReplaceTempView('ua')
t = time.time()
roofs = sedona.sql(f"""
    SELECT p.id, p.district, p.suitable, p.p_suitable, CAST(p.area_type AS INT) AS area_type, p.area_m2,
           p.centre_lon, p.centre_lat, p.chm_max_15m,
           coalesce(u.kwp_usable, 0D)                                        AS kw_max,
           coalesce(u.kwh_year_usable, 0D)                                   AS kwh_max,
           least(coalesce(u.kwp_usable, 0D), {HOME_KW}D)                     AS kw_real,
           coalesce(least(coalesce(u.kwp_usable, 0D), {HOME_KW}D) * try_divide(p.kwh_year, p.kwp), 0D) AS kwh_real,
           lower(hex(ST_H3CellIDs(ST_Point(p.centre_lon, p.centre_lat), 7, false)[0])) AS h3
    FROM pred p LEFT JOIN ua u ON p.id = u.id
""").cache()
roofs.createOrReplaceTempView('roofs')
N = roofs.count()
log(f'Roofs: {N:,} ({time.time() - t:.0f} s)')

OUTDIR = f'{LOCAL}/dashboard_data'
shutil.rmtree(OUTDIR, ignore_errors=True)
os.makedirs(OUTDIR)
SUMS = """count(*) AS roofs, sum(CAST(suitable AS INT)) AS suitable_roofs,
           round(100 * avg(CAST(suitable AS DOUBLE)), 1) AS pct_suitable"""

hexes = sedona.sql(f"""
    SELECT h3, mode(district) AS district, {SUMS},
           round(sum(kw_max) / 1e3, 2) AS mw_max, round(sum(kwh_max) / 1e6, 2) AS gwh_max,
           round(sum(kw_real) / 1e3, 2) AS mw_real, round(sum(kwh_real) / 1e6, 2) AS gwh_real,
           round(avg(chm_max_15m), 1) AS avg_tree_m,
           round(avg(centre_lat), 5) AS lat, round(avg(centre_lon), 5) AS lon
    FROM roofs GROUP BY h3
""").toPandas()
hexes.to_csv(f'{OUTDIR}/hex7.csv', index=False)
log(f'Hexagons (H3 res 7): {len(hexes):,}')

district = sedona.sql(f"""
    SELECT district, {SUMS}, round(avg(area_m2), 1) AS avg_roof_m2,
           round(sum(kw_max) / 1e6, 2) AS gw_max, round(sum(kwh_max) / 1e9, 2) AS twh_max,
           round(sum(kw_real) / 1e6, 2) AS gw_real, round(sum(kwh_real) / 1e9, 2) AS twh_real,
           round(100 * avg(CAST(area_type = 1 AS DOUBLE)), 1) AS pct_town_roofs,
           round(avg(chm_max_15m), 1) AS avg_tree_m
    FROM roofs GROUP BY district ORDER BY twh_real DESC
""").toPandas()
district.to_csv(f'{OUTDIR}/district.csv', index=False)
log('\nDISTRICTS (check: totals should match notebook 11: 64.29 GW maximum, 18.50 GW realistic)')
log_table(district)
log(f"KERALA: {district.roofs.sum():,} roofs | max {district.gw_max.sum():.2f} GW / {district.twh_max.sum():.2f} TWh | "
    f"realistic {district.gw_real.sum():.2f} GW / {district.twh_real.sum():.2f} TWh")

sedona.sql(f"""
    SELECT area_type, {SUMS}, round(sum(kw_max) / 1e6, 2) AS gw_max, round(sum(kw_real) / 1e6, 2) AS gw_real,
           round(sum(kwh_real) / 1e9, 2) AS twh_real
    FROM roofs GROUP BY area_type ORDER BY area_type
""").toPandas().to_csv(f'{OUTDIR}/area_type.csv', index=False)

sample = (roofs.sample(fraction=min(1.0, 40000 / N), seed=7)
          .selectExpr('district', 'round(centre_lon, 6) AS lon', 'round(centre_lat, 6) AS lat', 'suitable',
                      'round(p_suitable, 2) AS p_suitable', 'round(area_m2) AS area_m2', 'round(kw_max, 1) AS kw_max')
          .toPandas())
sample.to_csv(f'{OUTDIR}/roof_sample.csv', index=False)
log(f'Roof sample: {len(sample):,} roofs')

In [ ]:
# Step 5: simplified district borders (Sedona) + the CSV results of the earlier notebooks, zipped
import glob
rows = (sedona.read.format('geoparquet').load(f'{LOCAL}/kerala_boundaries.parquet').filter("subtype = 'county'")
        .selectExpr('name AS district', 'ST_AsGeoJSON(ST_SimplifyPreserveTopology(geometry, 0.002)) AS gj').collect())
fc = {'type': 'FeatureCollection',
      'features': [{'type': 'Feature', 'properties': {'district': r['district']}, 'geometry': json.loads(r['gj'])}
                   for r in rows]}
with open(f'{OUTDIR}/districts.geojson', 'w') as f:
    json.dump(fc, f)
log(f'District borders: {len(rows)} | names match the roof data: '
    f'{sorted(r["district"] for r in rows) == sorted(district.district)}')

for path in glob.glob(f'{RESULTS}/*.csv'):
    shutil.copy(path, OUTDIR)
for name in glob.glob(f'{LOGS}/*.txt'):
    shutil.copy(name, OUTDIR)
zip_path = shutil.make_archive(f'{LOCAL}/dashboard_data', 'zip', OUTDIR)
shutil.copy(zip_path, f'{RESULTS}/dashboard_data.zip')
size = os.path.getsize(zip_path) / 1e6
log(f'\nSaved {RESULTS}/dashboard_data.zip ({size:.1f} MB, {len(os.listdir(OUTDIR))} files)')
log('ALL DONE')

## Done

1. Download `MyDrive/rooftop_solar/results/dashboard_data.zip`.
2. Unzip it into your project folder as `dashboard\data` (so you have `dashboard\data\hex7.csv`, …).
3. Send Claude the text of `logs/12_log.txt`.